# Week 07 Wednesday · NLP Take-Home Assignment
## Hard NLP Patterns & Aspect-Based Sentiment Analysis

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from src.sentiment_patterns import analyze_patterns, get_baseline_failure_mode
from src.aspect_extractor import extract_aspect_sentiment, explain_aspect_hardness, get_improvement_strategies

# Load Wednesday Dataset
try:
    df = pd.read_csv('../data/ShopSense_Reviews_Wednesday.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Dataset not found. Please run data_generator.py first.")

df.head()

Dataset loaded successfully.


,review_id,review_text,pattern,category,rating
0,0,this product is not bad at all,Negation,Home,2
1,1,Wow great! Broke on day 1,Sarcasm,Home,2
2,2,Product bahut accha hai lekin delivery late thi,Code-mixing,Food,4
3,3,Returned it within 2 hours,Implicit,Home,5
4,4,Way better than my previous Samsung,Comparative,Clothing,3


## Q1: Handling 5 Hardest NLP Patterns
We demonstrate how specific patterns (Negation, Sarcasm, Code-mixing, Implicit, Comparative) manifest and why baseline models typically fail.

In [2]:
# Filter reviews with specific patterns
pattern_reviews = df[df['pattern'] != 'Basic'].copy()

results = []
for idx, row in pattern_reviews.iterrows():
    text = row['review_text']
    detected = analyze_patterns(text)
    primary_pattern = detected[0] if detected else "None"
    failure = get_baseline_failure_mode(primary_pattern)
    
    results.append({
        "Review": text,
        "Pattern": primary_pattern,
        "Baseline Failure Mode": failure
    })

pd.DataFrame(results).style.set_properties(**{'text-align': 'left'})

,Review,Pattern,Baseline Failure Mode
0,this product is not bad at all,Negation,"Standard BOW sees 'not' as negative, missing the flip from 'at all' making it positive."
1,Wow great! Broke on day 1,Sarcasm,"Bag-of-words weights 'great' as positive, missing the logical contradiction of 'Broke'."
2,Product bahut accha hai lekin delivery late thi,Code-mixing,Standard English tokenizers and lexicons (VADER) treat Hindi words as unknown/neutral (0 score).
3,Returned it within 2 hours,Implicit,Sentiment lexicons often miss actions (verb phrases) like 'Returned' which carry massive negative signal.
4,Way better than my previous Samsung,Comparative,Models may score both brands (e.g. Samsung) without understanding the relative preference direction.
5,Amazing camera quality but the battery is atrocious and customer support was unhelpful.,None,General semantic gap.
6,"The fabric is soft and comfortable, but the stitching is coming loose after one wash.",None,General semantic gap.
7,"Delicious food and great ambiance, however the service was extremely slow.",None,General semantic gap.


## Q2 (a) & (b): Aspect-Level Classification vs Review-Level

In [3]:
print("--- Why is Aspect-Level Harder? ---")
for reason, detail in explain_aspect_hardness().items():
    print(f"- {reason}: {detail}")

print("\n--- How to improve from 71% to 80%+? ---")
for i, strategy in enumerate(get_improvement_strategies()):
    print(f"{i+1}. {strategy}")

--- Why is Aspect-Level Harder? ---
- Data Sparsity: Aspect-level requires labeling every entity, not just the whole doc.
- Sentiment Ambiguity: The same word (e.g., 'hot') can be positive for food but negative for a phone.
- Coreference Resolution: Identifying that 'it' refers to the 'camera' and not the 'case' in the next sentence.
- Overlapping Sentiments: A single sentence can praise one aspect while criticizing another.

--- How to improve from 71% to 80%+? ---
1. Dependency Parsing: Linking adjectives to specifically modified nouns.
2. Domain-Specific Embeddings: Training on e-commerce data to capture product-specific jargon.
3. Transfer Learning: Using BERT/RoBERTa focused on Aspect-Based Sentiment Analysis (ABSA).
4. Data Augmentation: Synthesizing more balanced examples for rare aspects.


## Q2 (c): Extracting Aspect-Sentiment Pairs
Demonstrating how a single review can contain multiple conflicting sentiments aligned to different product traits.

In [4]:
test_rev = "Amazing camera quality but the battery is atrocious and customer support was unhelpful."
aspects = extract_aspect_sentiment(test_rev)

print(f"Original Review: '{test_rev}'")
print("\nExtracted Aspects:")
for item in aspects:
    print(f" -> Aspect: {item['aspect']:<10} | Sentiment: {item['sentiment']}")

Original Review: 'Amazing camera quality but the battery is atrocious and customer support was unhelpful.'

Extracted Aspects:
 -> Aspect: camera     | Sentiment: Positive
 -> Aspect: battery    | Sentiment: Negative
 -> Aspect: support    | Sentiment: Negative


### Conclusion:
As shown above, the review is simultaneously **Positive** (on Camera) and **Negative** (on Battery and Support). Review-level sentiment (e.g. 'Negative') would lose the critical praise for the hardware, which is why product teams prioritize aspect-level analysis.